# WMSD Case Studies

In this notebook, we reproduce the case studies from the paper _"Towards Explainable TOPSIS: Visual Insights into the Effects of Weights and Aggregations on Rankings"_. The case studies were conducted on a dataset of students described in terms of school grades and on a dataset of countries described in terms of factors constituting the Index of Economic Freedom. The datasets can be found in the data folder in this repository.

## Student grades

The first dataset contains 15 alternatives, i.e., students described by three criteria which are the average grades obtained by these students in Maths, Biology, and Art. The domains of the criteria are [0,100] for Maths, [1,6] for Biology, and [1,6] for Art. The dataset is loaded and visualized below.

In [ ]:
import pandas as pd
from pathlib import Path
from wmsd import WMSDTransformer

repo_root = Path.cwd()
if not (repo_root / 'data' / 'students.csv').exists():
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / 'data' / 'students.csv', sep=';', index_col=0)
students_transformer = WMSDTransformer('R')  # Relative distance
students_transformer.fit_transform(df, expert_range=[[0, 100], [1, 6], [1, 6]])


In [ ]:
df_students = pd.read_csv(repo_root / 'data' / 'wmsd_students.csv', index_col=0)
df_students.style.format(precision=2)


First, we will process the dataset using uniform criteria weights. And present the values of the three analyzed aggregation functions (I, A, R).

In [ ]:
from wmsd import WMSDTransformer

grade_ranges = [[0, 100], [1, 6], [1, 6]]

student_transformer = WMSDTransformer('I')
i = student_transformer.fit_transform(df_students, expert_range=grade_ranges).I
student_transformer = WMSDTransformer('A')
a = student_transformer.fit_transform(df_students, expert_range=grade_ranges).A
student_transformer = WMSDTransformer('R')
r = student_transformer.fit_transform(df_students, expert_range=grade_ranges).R

df_students_nw = pd.DataFrame({'I': i, 'A': a, 'R': r}, index=df_students.index)
df_students_nw.style.format(precision=2)


MSD-space for the R aggregation looks as follows.

In [ ]:
fig = student_transformer.plot()
fig

Perfoming the same operations for weights w = [0.5, 0.6, 1.0], we get the results below.

In [ ]:
w = [0.5, 0.6, 1.0]

student_transformer = WMSDTransformer("I")
i = student_transformer.fit_transform(df_students, expert_range=grade_ranges, weights=w).I
student_transformer = WMSDTransformer("A")
a = student_transformer.fit_transform(df_students, expert_range=grade_ranges, weights=w).A
student_transformer = WMSDTransformer("R")
r = student_transformer.fit_transform(df_students, expert_range=grade_ranges, weights=w).R

df_students_nw = pd.DataFrame({"I": i, "A": a, "R": r}, index=df_students.index)
df_students_nw.style.format(precision=2)

In [ ]:
fig = student_transformer.plot()
fig

## Index of Economic Freedom

The second case study is based on publicly available data from the [Index of Economic Freedom](https://www.heritage.org/index/download), which covers 12 freedoms - from property rights to tax burdens - in 184 countries. The data has been annually collected for almost 30 years now by The Heritage Foundation. In particular, our case study is based on the data gathered for the 25th anniversary of the Index in 2019.

Economic freedom is understood as the right of every human to control their own labor and property.
Within the Index, 12 factors are measured and grouped into four categories: Rule of Law, Government Size, Regulatory Efficiency, Open Markets. There are three factors per category, each factor is graded on a 0-100 scale of type gain. For the purpose of this case study, we have limited the Index only to the 12 countries of South America and aggregated the criteria by taking the mean of factors forming each category. The dataset is loaded and visualized below.

In [ ]:
south_america = ['Argentina', 'Bolivia', 'Brazil', 'Chile',
                 'Colombia', 'Ecuador', 'Guyana', 'Paraguay',
                 'Peru', 'Suriname', 'Uruguay', 'Venezuela']

df_full = pd.read_csv(repo_root / 'data' / 'index_2019.csv')
df_sa = df_full[df_full['Country Name'].isin(south_america)]
df_sa.style.format(precision=2)


As mentioned above, we will aggregate the criteria by taking the mean of factors forming each category. The aggregated dataset, our decision matrix, is presented below.

In [ ]:
rule_of_law = (df_sa.iloc[:, 7] + df_sa.iloc[:, 8] + df_sa.iloc[:, 9]) / 3
govt_size = (df_sa.iloc[:, 10] + df_sa.iloc[:, 11] + df_sa.iloc[:, 12]) / 3
regulatory_efficiency = (df_sa.iloc[:, 13] + df_sa.iloc[:, 14] + df_sa.iloc[:, 15]) / 3
open_markets = (df_sa.iloc[:, 16] + df_sa.iloc[:, 17] + df_sa.iloc[:, 18]) / 3


df_sa_cs = pd.DataFrame({"Country": df_sa["Country Name"],
                         "Rule of Law": rule_of_law,
                         "Government size": govt_size,
                         "Regulatory efficiency": regulatory_efficiency,
                         "Open Markets": open_markets})
df_sa_cs.set_index("Country", inplace=True)
df_sa_cs.style.format(precision=2)

Below we present the aggregation values and WMSD-spaces for the four different weight vectors considered in the paper.

### w1 = [1.00, 1.00, 1.00, 1.00]

In [ ]:
sa_transformer = WMSDTransformer("R")
criteria_ranges = [[0,100],[0,100],[0,100],[0,100]]
w1 = [1.00, 1.00, 1.00, 1.00]

df_sa_w1 = sa_transformer.fit_transform(df_sa_cs, expert_range=criteria_ranges, weights=w1)
df_sa_w1 = df_sa_w1.sort_values(by="R", ascending=False)
df_sa_w1.style.format(precision=3)

In [ ]:
fig = sa_transformer.plot()
fig

### w2 = [0.25, 1.00, 0.25, 0.50]

In [ ]:
w2 = [0.25, 1.00, 0.25, 0.50]
df_sa_w2 = sa_transformer.fit_transform(df_sa_cs, expert_range=criteria_ranges, weights=w2)
df_sa_w2 = df_sa_w2.sort_values(by="R", ascending=False)
df_sa_w2.style.format(precision=3)

In [ ]:
fig = sa_transformer.plot()
fig

### w3 = [0.50, 1.00, 0.25, 0.25]

In [ ]:
w3 = [0.50, 1.00, 0.25, 0.25]
df_sa_w3 = sa_transformer.fit_transform(df_sa_cs, expert_range=criteria_ranges, weights=w3)
df_sa_w3 = df_sa_w3.sort_values(by="R", ascending=False)
df_sa_w3.style.format(precision=3)

In [ ]:
fig = sa_transformer.plot()
fig

### w4 = [1.00, 2/3, 1/3, 0.00]

In [ ]:
w4 = [1.00, 2/3, 1/3, 0.00]
df_sa_w4 = sa_transformer.fit_transform(df_sa_cs, expert_range=criteria_ranges, weights=w4)
df_sa_w4 = df_sa_w4.sort_values(by="R", ascending=False)
df_sa_w4.style.format(precision=3)

In [ ]:
fig = sa_transformer.plot()
fig